# Previsão de Rotatividade de Funcionários (Turnover)
## Machine Learning - Classificação Binária

**Problema:** Prever se um funcionário tem alta probabilidade de deixar a empresa.

**Dataset:** IBM HR Analytics Employee Attrition & Performance (1470 registros, 35 atributos)

**Algoritmos:** Regressão Logística vs Random Forest

**Métrica principal:** F1-Score (dado o desbalanceamento das classes)

## 1. Importação das Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

sns.set_style('whitegrid')
RANDOM_STATE = 42
print('Bibliotecas importadas com sucesso!')

## 2. Carregamento e Análise Exploratória (EDA)

In [2]:
df = pd.read_csv('../data/HR_Employee_Attrition.csv', encoding='utf-8-sig')
print(f'Shape: {df.shape}')
print(f'\nPrimeiras linhas:')
display(df.head())

In [3]:
print('Info do Dataset:')
df.info()

In [4]:
valores_nulos = df.isnull().sum().sum()
print(f'Valores nulos: {valores_nulos}')

attrition_dist = df['Attrition'].value_counts()
attrition_pct = df['Attrition'].value_counts(normalize=True) * 100
print(f'\nDistribuição Attrition:')
print(attrition_dist)
print(f'\nPercentual:')
print(attrition_pct.round(2))

### 2.1 Visualizações Exploratórias

In [5]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Análise Exploratória de Dados', fontsize=16, fontweight='bold')

sns.countplot(data=df, x='Attrition', hue='Attrition', ax=axes[0,0], legend=False)
axes[0,0].set_title('Distribuição de Attrition')

sns.boxplot(data=df, x='Attrition', y='Age', hue='Attrition', ax=axes[0,1], legend=False)
axes[0,1].set_title('Idade vs Attrition')

sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', hue='Attrition', ax=axes[0,2], legend=False)
axes[0,2].set_title('Renda Mensal vs Attrition')

att_by_dept = df.groupby('Department')['Attrition'].value_counts(normalize=True).unstack() * 100
att_by_dept.plot(kind='bar', stacked=True, ax=axes[1,0], color=['#4CAF50','#F44336'])
axes[1,0].set_title('Attrition por Departamento (%)')
axes[1,0].set_ylabel('Percentual')
axes[1,0].legend(title='Attrition')

sns.countplot(data=df, x='OverTime', hue='Attrition', ax=axes[1,1])
axes[1,1].set_title('Hora Extra vs Attrition')

sns.countplot(data=df, x='JobRole', hue='Attrition', ax=axes[1,2])
axes[1,2].set_title('Attrition por Cargo')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../slides/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pré-processamento

Removendo colunas sem variância (EmployeeCount, StandardHours, Over18) e o identificador (EmployeeNumber).
Utilizando ColumnTransformer com Pipeline para evitar data leakage.

In [6]:
colunas_remover = ['Attrition', 'EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
X = df.drop(columns=colunas_remover)
y = df['Attrition'].map({'Yes': 1, 'No': 0})

print(f'Features preditoras: {X.shape[1]}')
print(f'\nFeatures numéricas:')
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(num_features)
print(f'\nFeatures categóricas:')
cat_features = X.select_dtypes(include=['object']).columns.tolist()
print(cat_features)

In [7]:
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

print('ColumnTransformer criado com sucesso!')

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Treino: {X_train.shape}')
print(f'Teste:  {X_test.shape}')
print(f'\nDistribuição Attrition - Treino:')
print(y_train.value_counts(normalize=True).mul(100).round(2))
print(f'\nDistribuição Attrition - Teste:')
print(y_test.value_counts(normalize=True).mul(100).round(2))

## 4. Treinamento dos Modelos

Dois modelos serão treinados e otimizados via GridSearchCV:
1. **Regressão Logística** (com class_weight='balanced')
2. **Random Forest** (com class_weight='balanced')

In [9]:
models = {
    'Regressao Logistica': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', random_state=RANDOM_STATE
    ),
}

param_grids = {
    'Regressao Logistica': {
        'classifier__C': [0.01, 0.1, 1, 10, 100],
        'classifier__penalty': ['l2'],
    },
    'Random Forest': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [5, 10, None],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
    },
}

results = []
best_models = {}

for name, model in models.items():
    print(f'\n{"="*50}')
    print(f'>>> Treinando: {name}')
    print(f'{"="*50}')
    
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model),
    ])
    
    gs = GridSearchCV(
        pipe, param_grids[name],
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring='f1', n_jobs=-1, verbose=1
    )
    gs.fit(X_train, y_train)
    
    y_pred = gs.predict(X_test)
    y_proba = gs.predict_proba(X_test)[:, 1]
    
    best_models[name] = gs.best_estimator_
    
    results.append({
        'model': name,
        'best_params': gs.best_params_,
        'best_score': gs.best_score_,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'y_pred': y_pred,
        'y_proba': y_proba,
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'report': classification_report(y_test, y_pred),
    })
    
    if name == 'Random Forest':
        rf_model = gs.best_estimator_.named_steps['classifier']
        results[-1]['feature_importances'] = rf_model.feature_importances_

## 5. Resultados dos Modelos

In [10]:
for r in results:
    print(f'\n{"="*50}')
    print(f"Modelo: {r['model']}")
    print(f'{"="*50}')
    print(f"Melhores hiperparametros: {r['best_params']}")
    print(f"Melhor F1 (CV): {r['best_score']:.4f}")
    print(f"Acuracia (Teste):  {r['accuracy']:.4f}")
    print(f"Precisao:  {r['precision']:.4f}")
    print(f"Recall:    {r['recall']:.4f}")
    print(f"F1-Score:  {r['f1_score']:.4f}")
    print(f"ROC-AUC:   {r['roc_auc']:.4f}")
    print(f"\nRelatorio:\n{r['report']}")

### 5.1 Comparação Visual

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparacao de Modelos - Previsao de Turnover', fontsize=16, fontweight='bold')

metrics_df = pd.DataFrame([
    {'Model': r['model'], 'Acuracia': r['accuracy'],
     'Precisao': r['precision'], 'Recall': r['recall'],
     'F1-Score': r['f1_score'], 'ROC-AUC': r['roc_auc']}
    for r in results
]).set_index('Model')

metrics_df.plot(kind='bar', ax=axes[0,0], rot=0, colormap='viridis')
axes[0,0].set_title('Metricas de Desempenho')
axes[0,0].set_ylim(0, 1)
axes[0,0].legend(loc='lower right')
axes[0,0].grid(axis='y', alpha=0.3)

for r in results:
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    axes[0,1].plot(fpr, tpr, label=f"{r['model']} (AUC={r['roc_auc']:.3f})")
axes[0,1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0,1].set_title('Curva ROC')
axes[0,1].set_xlabel('Taxa de Falso Positivo')
axes[0,1].set_ylabel('Taxa de Verdadeiro Positivo')
axes[0,1].legend()
axes[0,1].grid(alpha=0.3)

for i, r in enumerate(results):
    sns.heatmap(r['confusion_matrix'], annot=True, fmt='d',
                cmap='Blues', ax=axes[1,i])
    axes[1,i].set_title(f"Matriz de Confusao - {r['model']}")
    axes[1,i].set_xlabel('Previsto')
    axes[1,i].set_ylabel('Real')

plt.tight_layout()
plt.savefig('../slides/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Importância das Features (Random Forest)

In [12]:
rf_result = [r for r in results if r['model'] == 'Random Forest'][0]

if 'feature_importances' in rf_result:
    fi = rf_result['feature_importances']
    cat_feature_names = best_models['Random Forest'].named_steps['preprocessor']\
        .named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)
    all_feature_names = np.concatenate([num_features, cat_feature_names])
    
    fi_df = pd.DataFrame({'feature': all_feature_names, 'importance': fi})\
        .sort_values('importance', ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=fi_df, y='feature', x='importance', palette='viridis')
    plt.title('Top 15 Features mais Importantes - Random Forest', fontsize=14, fontweight='bold')
    plt.xlabel('Importancia')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.savefig('../slides/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    display(fi_df)

## 7. Conclusões

### Análise dos Resultados:

1. **Problema de desbalanceamento:** Aproximadamente 84% dos funcionários permanecem e apenas 16% saem.

2. **Métrica principal (F1-Score):** Devido ao desbalanceamento, o F1-Score é a métrica mais adequada, pois
   equilibra Precisão e Recall para a classe minoritária (Saiu da empresa = Sim).

3. **Regressão Logística vs Random Forest:**
   - A Regressão Logística oferece maior interpretabilidade (coeficientes lineares)
   - O Random Forest captura relações não-lineares complexas
   - Ambos usaram class_weight='balanced' para lidar com o desbalanceamento

4. **Principais preditores de turnover:** Horas extras (OverTime), Renda Mensal (MonthlyIncome),
   Distância de Casa (DistanceFromHome), e Anos na Empresa (YearsAtCompany).

### Próximos passos:
- Testar SMOTE como alternativa ao class_weight='balanced'
- Explorar outros algoritmos (SVM, XGBoost)
- Coletar mais dados para melhorar a generalização